# 04 · INT8 and NF4 for the Five Core Models

**Project:** Silent Tool-Call Failures: How Quantization Degrades Function Calling in Small LLMs
**Update 5 goal:** the accuracy of every core model in two bitsandbytes precisions, INT8 and NF4, compared with the FP16 baselines from Update 4.

| Precision | Method | How it is made |
|---|---|---|
| INT8 | LLM.int8() (bitsandbytes): 8-bit weights with one scale per row; input features above 6.0 stay in FP16 | vLLM cannot quantize to 8 bits while loading, so `src/quantize.py` first makes an 8-bit checkpoint with transformers. vLLM runs it in eager mode (no CUDA graphs). |
| NF4 | 4-bit NormalFloat (bitsandbytes), blocks of 64, double quantization | vLLM quantizes the FP16 weights while loading |

Everything else is the same as in Update 4: the 1,390 BFCL `non_live` questions, the prompt, greedy decoding and the scorer (`src/run_eval.py`, `src/score.py`). Every run uses **one** T4, Qwen3-8B too (in FP16 it needed both).

**Quick check first:** Qwen3-1.7B answers 2 questions per category in INT8 and in NF4. If a precision fails there, its runs are skipped and the other precision still runs.

## How to run on Kaggle

1. kaggle.com → **Create → New Notebook → File → Import Notebook**, then upload this file.
2. In the right panel, under **Settings**: **Accelerator → GPU T4 x2** and **Internet → On**.
3. **Required for Llama:** **Add-ons → Secrets** → add `HF_TOKEN` (a Hugging Face *read* token from huggingface.co → Settings → Access Tokens) and switch it on for this notebook.
4. Click **Run All** to watch it, or **Save Version → Save & Run All (Commit)** to let it run in the background.

**Time:** probably 3-6 hours. This is a guess: it is the first quantized run, and INT8 is slower than FP16 (eager mode and the outlier step). The quick check prints the speed early. Kaggle stops a session after 12 hours.
**Output:** `update5_results.zip` with the raw answers, scores, run information, logs and `int8_nf4.md`.

## 1. Settings
Edit this cell only.

In [1]:
import os

REPO_URL = "https://github.com/shawalkabirchy/quantization-functional-call"
REPO_REF = "main"             # branch, tag or commit of the code to run

# Grouped by model (INT8, then NF4), so each model is downloaded once; Qwen3-8B last
CONFIGS = ["configs/qwen3-8b_nf4.yaml"]
CHECK_CONFIGS = []
RUN_TIMEOUT_MIN = 200
CHECK_LIMIT = 2               # questions per category in the quick check
MODE = "free"
REPORT_NAME = "int8_nf4.md"
REPORT_TITLE = "INT8 and NF4 (Update 5)"        # a run that takes longer than this is stopped, and the next one starts
QUANTIZE_TIMEOUT_MIN = 30     # the same for making an INT8 checkpoint
DELETE_MODEL_AFTER_RUN = True # frees disk space: deletes each INT8 checkpoint after its run, each model after its last run

BFCL_VERSION = "2026.3.23"    # pinned for reproducibility
BITSANDBYTES_VERSION = "0.45.5"  # vLLM 0.8.5 needs at least 0.45.3
ACCELERATE_VERSION = "1.6.0"     # transformers needs it to load a model in 8 bits
WORK = "/kaggle/working"
REPO = f"{WORK}/repo"                         # code and results (kept as notebook output)
VENV = os.path.expanduser("~/bfcl-venv")      # separate Python 3.11 environment (not kept)
CHECKPOINT_DIR = os.path.expanduser("~/checkpoints")   # INT8 checkpoints (not kept: they are large)
os.environ["CHECKPOINT_DIR"] = CHECKPOINT_DIR           # read by src/quantize.py and src/run_eval.py


def run_id_of(config):
    return os.path.splitext(os.path.basename(config))[0] + f"_{MODE}"

## 2. Check the GPU, internet and Hugging Face token

In [2]:
import json
import shutil
import subprocess
import time
from datetime import date

import requests

smi = subprocess.run(
    ["nvidia-smi", "--query-gpu=index,name,memory.total,driver_version", "--format=csv,noheader"],
    capture_output=True, text=True,
)
if smi.returncode != 0:
    raise RuntimeError("No GPU found. Set Settings -> Accelerator -> GPU T4 x2, then run again.")
print("GPUs:\n" + smi.stdout)

try:
    requests.get("https://pypi.org/simple/", timeout=15).raise_for_status()
    print("Internet: OK")
except Exception as e:
    raise RuntimeError("No internet. Set Settings -> Internet -> On (needs a phone-verified Kaggle account).") from e

try:
    from kaggle_secrets import UserSecretsClient
    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
    print("HF_TOKEN: loaded from Kaggle Secrets")
except Exception:
    print("HF_TOKEN: NOT SET. Llama 3.2 cannot be downloaded without it (Add-ons -> Secrets).")

GPUs:
0, Tesla T4, 15360 MiB, 580.178.04
1, Tesla T4, 15360 MiB, 580.178.04

Internet: OK
HF_TOKEN: NOT SET. Llama 3.2 cannot be downloaded without it (Add-ons -> Secrets).


## 3. Get the code and check access to every model
Clones the repository, then downloads one small file of each model at the pinned revision. This finds a missing token or missing Llama approval now, not after an hour of running.

In [3]:
import yaml
from huggingface_hub import hf_hub_download

shutil.rmtree(REPO, ignore_errors=True)
!git clone --quiet {REPO_URL} {REPO}
!git -C {REPO} checkout --quiet {REPO_REF}
!git -C {REPO} log -1 --format="Code version: %h %s (%ci)"

cfgs = {}
problems = []
for config in CONFIGS:
    with open(f"{REPO}/{config}") as f:
        cfgs[config] = yaml.safe_load(f)
    model, revision = cfgs[config]["model"], str(cfgs[config]["revision"])
    try:
        hf_hub_download(model, "config.json", revision=revision, token=os.environ.get("HF_TOKEN"))
        print(f"  access OK  {model}")
    except Exception as e:
        problems.append(f"{model}: {type(e).__name__}: {str(e)[:200]}")
if problems:
    print("\n".join(problems))
    raise RuntimeError(
        "Some models cannot be downloaded. For Llama: add HF_TOKEN under Add-ons -> Secrets, switch it on for "
        "this notebook, and check that https://huggingface.co/meta-llama/Llama-3.2-1B-Instruct says you have access."
    )

Code version: 6b6903c Qwen3-8B INT8: gpu_memory_utilization 0.81 (0.90 ran out of memory) (2026-10-29 20:39:59 +0600)


config.json:   0%|          | 0.00/728 [00:00<?, ?B/s]

  access OK  Qwen/Qwen3-8B


## 4. Install BFCL, vLLM and bitsandbytes (about 1-5 minutes)
The same environment as notebooks 01-03 (a separate Python 3.11 environment with `bfcl-eval[oss_eval_vllm]`, vLLM 0.8.5, and the same pins), plus bitsandbytes and accelerate for the quantized models.

In [4]:
!pip install -q uv
shutil.rmtree(VENV, ignore_errors=True)
!uv venv {VENV} --python 3.11
!uv pip install --python {VENV}/bin/python "bfcl-eval[oss_eval_vllm]=={BFCL_VERSION}" "transformers==4.51.3" "openai>=1.86.0,<2" "fastapi<0.116" "pydantic<2.12" "setuptools>=77.0.3,<80" soundfile "bitsandbytes=={BITSANDBYTES_VERSION}" "accelerate=={ACCELERATE_VERSION}"

CHECK = (
    "import vllm, bfcl_eval, yaml, xgrammar, triton, torch, bitsandbytes, accelerate; "
    "from importlib.metadata import version; "
    "print('vllm', version('vllm'), '| xgrammar', version('xgrammar'), '| torch', version('torch'), "
    "'| bitsandbytes', version('bitsandbytes'), '| accelerate', version('accelerate')); "
    "print('GPUs:', [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])"
)
r = subprocess.run([f"{VENV}/bin/python", "-c", CHECK], capture_output=True, text=True)
print(r.stdout)
if r.returncode != 0:
    print(r.stderr[-5000:])
    raise RuntimeError("Environment check failed (see above). Check the output of the install step.")

freeze = subprocess.run(["uv", "pip", "freeze", "--python", f"{VENV}/bin/python"], capture_output=True, text=True)
with open(f"{WORK}/requirements-freeze.txt", "w") as f:
    f.write(freeze.stdout)
print(f"Saved {len(freeze.stdout.splitlines())} package versions to {WORK}/requirements-freeze.txt")

Using CPython 3.11.17
Creating virtual environment at: /root/bfcl-venv
Activate with: source /root/bfcl-venv/bin/activate
Using Python 3.11.17 environment at: /root/bfcl-venv
Resolved 215 packages in 4.07s
Prepared 215 packages in 59.05s
Installed 215 packages in 490ms
 + accelerate==1.6.0
 + agent-detector==2.0.0
 + aiohappyeyeballs==2.7.1
 + aiohttp==3.14.5
 + aiosignal==1.4.0
 + airportsdata==20260905
 + annotated-doc==0.0.5
 + annotated-types==0.8.0
 + anthropic==2.3.0
 + anyio==4.15.1
 + argcomplete==3.7.2
 + astor==0.8.1
 + attrs==26.1.0
 + beautifulsoup4==4.15.0
 + bfcl-eval==2026.3.23
 + bitsandbytes==0.45.5
 + black==26.10.1
 + blake3==1.0.11
 + boto3==1.43.121
 + botocore==1.43.121
 + cachetools==7.2.7
 + certifi==2026.10.23
 + cffi==2.1.1
 + charset-normalizer==3.5.2
 + click==8.5.0
 + cloudpickle==3.1.2
 + cohere==5.18.0
 + compressed-tensors==0.9.3
 + cryptography==50.0.2
 + cupy-cuda12x==13.6.0
 + dashscope==2.0.2
 + datamodel-code-generator==0.25.7
 + deprecated==1.3.1
 

## 5. Quick check: does each precision load and answer?
Qwen3-1.7B in INT8 (the checkpoint is made first) and in NF4, 2 questions per category. The answers go to `check/`, apart from the real results. A precision that fails here is skipped in step 6.

In [5]:
def tail(path, n=25):
    with open(path, errors="replace") as f:
        return "".join(f.readlines()[-n:])


def run_logged(args, log_path, timeout_min):
    """Runs a script of the repository in the BFCL environment; returns its exit code, or None after a timeout."""
    with open(log_path, "w") as log:
        try:
            return subprocess.run([f"{VENV}/bin/python", *args], cwd=REPO, stdout=log, stderr=subprocess.STDOUT,
                                  timeout=timeout_min * 60).returncode
        except subprocess.TimeoutExpired:
            print(f"  stopped after {timeout_min} minutes")
            return None


def gpu_memory_used_mib():
    out = subprocess.run(["nvidia-smi", "--query-gpu=memory.used", "--format=csv,noheader,nounits"],
                         capture_output=True, text=True).stdout
    return [int(x) for x in out.split()]


def wait_for_free_gpus(limit_mib=1000, wait_s=120):
    """A finished run should leave the GPUs empty; wait briefly if they are not yet."""
    start = time.time()
    while max(gpu_memory_used_mib()) > limit_mib and time.time() - start < wait_s:
        time.sleep(5)
    used = gpu_memory_used_mib()
    if max(used) > limit_mib:
        print(f"  WARNING: GPU memory still in use before this run: {used} MiB")


CHECK_DIR = f"{WORK}/check"
shutil.rmtree(CHECK_DIR, ignore_errors=True)
works = {}
for config in CHECK_CONFIGS:
    cfg, stem = cfgs[config], os.path.splitext(os.path.basename(config))[0]
    print(f"\n=== check {stem}: started {time.strftime('%H:%M:%S')} UTC", flush=True)
    wait_for_free_gpus()
    ok = True
    if "quantize" in cfg:
        log_path = f"{WORK}/check_quantize_{stem}.log"
        ok = run_logged(["src/quantize.py", "--config", config], log_path, QUANTIZE_TIMEOUT_MIN) == 0
        print("  checkpoint: " + tail(log_path, 1).strip())
        wait_for_free_gpus()
    if ok:
        log_path = f"{WORK}/check_{stem}.log"
        ok = run_logged(["src/run_eval.py", "--config", config, "--mode", MODE, "--limit", str(CHECK_LIMIT),
                         "--results-dir", CHECK_DIR, "--overwrite"], log_path, 30) == 0
    works[cfg["precision"]] = ok
    if not ok:
        print(f"  FAILED. End of {log_path}:\n{tail(log_path, 40)}")
        continue

    run_id = f"{run_id_of(config)}_limit{CHECK_LIMIT}"
    r = subprocess.run([f"{VENV}/bin/python", "src/score.py", f"{CHECK_DIR}/raw/{run_id}.jsonl"],
                       cwd=REPO, capture_output=True, text=True)
    print(r.stdout[-1500:])
    with open(f"{CHECK_DIR}/meta/{run_id}.json") as f:
        meta = json.load(f)
    print(f"  weights {meta['memory']['model_weights_gib']} GiB, {meta['output_tokens_per_second']} output tokens/s, "
          f"finish reasons {meta['finish_reasons']}")
    with open(f"{CHECK_DIR}/raw/{run_id}.jsonl") as f:
        for line in list(f)[:3]:
            row = json.loads(line)
            print(f"  {row['id']}: {row['output'][:200]!r}")

print("\nQuick check: " + ", ".join(f"{p.upper()} {'works' if ok else 'FAILED'}" for p, ok in works.items()))
if not all(works.values()):
    print("The runs of a failed precision are skipped in step 6; the other precision still runs.")


Quick check: 


## 6. Run and score every config (a few hours)
One config at a time: `quantize.py` (INT8 only: makes the checkpoint), `run_eval.py` (answers), then `score.py` (accuracy). Every step's full log is saved (`quantize_<config>.log`, `run_<run id>.log`). If a config fails, the notebook prints the end of its log and goes on with the next one.

In [6]:
status, minutes = {}, {}
last_config_of = {cfgs[c]["model"]: c for c in CONFIGS}   # each model is deleted after its last config
for config in CONFIGS:
    cfg, run_id = cfgs[config], run_id_of(config)
    stem = os.path.splitext(os.path.basename(config))[0]
    if not works.get(cfg["precision"], True):
        status[run_id] = "SKIPPED (quick check failed)"
    else:
        print(f"\n=== {run_id} ({cfg['model']}): started {time.strftime('%H:%M:%S')} UTC", flush=True)
        start = time.time()
        ok = True
        if "quantize" in cfg:
            wait_for_free_gpus()
            log_path = f"{WORK}/quantize_{stem}.log"
            ok = run_logged(["src/quantize.py", "--config", config], log_path, QUANTIZE_TIMEOUT_MIN) == 0
            if ok:
                print("  checkpoint: " + tail(log_path, 1).strip())
            else:
                status[run_id] = "QUANTIZE FAILED"
                print(f"  quantization FAILED. End of {log_path}:\n{tail(log_path)}")
        if ok:
            wait_for_free_gpus()
            log_path = f"{WORK}/run_{run_id}.log"
            run_logged(["src/run_eval.py", "--config", config, "--mode", MODE, "--overwrite"], log_path,
                       RUN_TIMEOUT_MIN)
            if not os.path.exists(f"{REPO}/results/meta/{run_id}.json"):
                status[run_id] = "FAILED"
                print(f"  FAILED. End of {log_path}:\n{tail(log_path)}")
            else:
                print("  answers: " + next((l.strip() for l in tail(log_path, 10).splitlines()
                                            if l.startswith("Done:")), ""))
                r = subprocess.run([f"{VENV}/bin/python", "src/score.py", f"results/raw/{run_id}.jsonl"],
                                   cwd=REPO, capture_output=True, text=True)
                print(r.stdout[-2000:])
                status[run_id] = "OK" if r.returncode == 0 else "SCORING FAILED"
                if r.returncode != 0:
                    print(r.stderr[-3000:])
        minutes[run_id] = (time.time() - start) / 60
        print(f"  {status[run_id]} after {minutes[run_id]:.1f} min")

    if DELETE_MODEL_AFTER_RUN:
        shutil.rmtree(f"{CHECKPOINT_DIR}/{stem}", ignore_errors=True)
        if last_config_of[cfg["model"]] == config:
            shutil.rmtree(os.path.expanduser(f"~/.cache/huggingface/hub/models--{cfg['model'].replace('/', '--')}"),
                          ignore_errors=True)

print("\nSummary:")
for run_id, s in status.items():
    took = f"  ({minutes[run_id]:.1f} min)" if run_id in minutes else ""
    print(f"  {run_id:28s} {s}{took}")
print(f"  total: {sum(minutes.values()) / 60:.1f} hours")
if any(s != "OK" for s in status.values()):
    print("Some runs did not finish. The results of the others are still reported and packaged below.")


=== qwen3-8b_nf4_free (Qwen/Qwen3-8B): started 18:22:46 UTC
  answers: Done: 1390 answers in 4895 s (14.7 output tokens/s); finish reasons {'stop': 1390}

qwen3-8b_nf4_free  (1390 questions)
  category               correct  accuracy   parsed
  simple_python          382/400     95.50   100.00
  simple_java             64/100     64.00   100.00
  simple_javascript        35/50     70.00    96.00
  multiple               190/200     95.00   100.00
  parallel               185/200     92.50   100.00
  parallel_multiple      176/200     88.00   100.00
  irrelevance            203/240     84.58   100.00
  simple_ast                         76.50         
  non_live_overall                   88.00         
  saved /kaggle/working/repo/results/scores/qwen3-8b_nf4_free.jsonl and /kaggle/working/repo/results/summary.csv

  OK after 86.4 min

Summary:
  qwen3-8b_nf4_free            OK  (86.4 min)
  total: 1.4 hours


## 7. Tables: FP16 vs. INT8 vs. NF4
The FP16 rows are the Update 4 results already in the repository. **Overall** is BFCL's Non-Live Overall Acc (mean of simple, multiple, parallel and parallel-multiple); irrelevance is reported separately. **Parsed** is the share of answers to the AST questions that were valid JSON in the requested format: answers that fail here are syntactic failures.

In [7]:
import re

import pandas as pd

COLUMNS = {
    "simple_python": "Python", "simple_java": "Java", "simple_javascript": "JavaScript",
    "multiple": "Multiple", "parallel": "Parallel", "parallel_multiple": "Parallel Multiple",
    "non_live_overall": "Overall", "irrelevance": "Irrelevance",
}
PRECISIONS = ["fp16", "int8", "nf4"]


def table_order(base):
    family = 0 if "qwen" in base else 1
    size = float(re.search(r"(\d+(?:\.\d+)?)b", base).group(1))
    return family, size


bases = sorted({os.path.basename(c).rsplit("_", 1)[0] for c in CONFIGS}, key=table_order)  # e.g. qwen3-1.7b
summary = pd.read_csv(f"{REPO}/results/summary.csv")
acc_rows, info_rows, overall, metas = [], [], {}, {}
for base in bases:
    for precision in PRECISIONS:
        run_id = f"{base}_{precision}_{MODE}"
        s = summary[summary["run_id"] == run_id].set_index("category")
        if s.empty or "non_live_overall" not in s.index:
            continue
        with open(f"{REPO}/results/meta/{run_id}.json") as f:
            meta = metas[run_id] = json.load(f)
        scores = pd.read_json(f"{REPO}/results/scores/{run_id}.jsonl", lines=True)
        ast = scores[scores["category"] != "irrelevance"]
        name, label = meta["model"].split("/")[-1], precision.upper()
        overall.setdefault(name, {})[label] = float(s.loc["non_live_overall", "accuracy"])
        acc_rows.append({"Model": name, "Precision": label,
                         **{col: float(s.loc[c, "accuracy"]) for c, col in COLUMNS.items()}})
        info_rows.append({
            "Model": name,
            "Precision": label,
            "Parsed, AST questions (%)": round(100 * ast["parse_ok"].mean(), 2),
            "Unparsed answers": int((~ast["parse_ok"]).sum()),
            "Cut off at max_tokens": meta["finish_reasons"].get("length", 0),
            "GPUs": meta["config"]["tensor_parallel_size"],
            "Weights per GPU (GiB)": meta["memory"]["model_weights_gib"],
            "Generation (s)": meta["generation_seconds"],
            "Output tokens/s": meta["output_tokens_per_second"],
        })

compare_rows = []
for name, acc in overall.items():
    row = {"Model": name, **{p.upper(): acc.get(p.upper()) for p in PRECISIONS}}
    for p in ["INT8", "NF4"]:
        row[f"{p} - FP16"] = round(acc[p] - acc["FP16"], 2) if p in acc and "FP16" in acc else None
    compare_rows.append(row)

compare = pd.DataFrame(compare_rows)
accuracy = pd.DataFrame(acc_rows)
info = pd.DataFrame(info_rows)
display(compare)
display(accuracy)
display(info)
compare.to_csv(f"{WORK}/int8_nf4_overall.csv", index=False)
accuracy.to_csv(f"{WORK}/int8_nf4_accuracy.csv", index=False)
info.to_csv(f"{WORK}/int8_nf4_run_info.csv", index=False)

if not info.empty:
    quantized = info[info["Precision"] != "FP16"]
    missing = quantized[quantized["Weights per GPU (GiB)"].isna() | quantized["Output tokens/s"].isna()]
    print("Memory and speed recorded for every quantized run." if missing.empty else
          f"WARNING: memory or speed missing for:\n{missing[['Model', 'Precision']]}")

,Model,FP16,INT8,NF4,INT8 - FP16,NF4 - FP16
0,Qwen3-8B,88.29,None,88.0,None,-0.29


,Model,Precision,Python,Java,JavaScript,Multiple,Parallel,Parallel Multiple,Overall,Irrelevance
0,Qwen3-8B,FP16,95.5,65.0,68.0,93.0,92.5,91.5,88.29,87.08
1,Qwen3-8B,NF4,95.5,64.0,70.0,95.0,92.5,88.0,88.00,84.58


,Model,Precision,"Parsed, AST questions (%)",Unparsed answers,Cut off at max_tokens,GPUs,Weights per GPU (GiB),Generation (s),Output tokens/s
0,Qwen3-8B,FP16,99.91,1,0,2,7.6394,401.5,178.9
1,Qwen3-8B,NF4,99.83,2,0,1,5.6681,4895.0,14.7


Memory and speed recorded for every quantized run.


## 8. Write the report and package the results

In [8]:
import glob


def md_table(df):
    def fmt(v):
        if v is None or (isinstance(v, float) and v != v):
            return ""
        return f"{v:.2f}" if isinstance(v, float) else str(v)
    head = "| " + " | ".join(df.columns) + " |"
    sep = "|" + "|".join("---" if i == 0 else "---:" for i in range(len(df.columns))) + "|"
    return [head, sep] + ["| " + " | ".join(fmt(v) for v in row) + " |" for row in df.itertuples(index=False)]


int8 = next((m for m in metas.values() if m["precision"] == "int8"), None)
first = next((m for m in metas.values() if m["precision"] != "fp16"), None)
lines = [f"# {REPORT_TITLE}", "", f"**Date:** {date.today()}",
         "**Goal:** function-calling accuracy of every core model in INT8 and NF4 (bitsandbytes), compared with "
         "the FP16 baselines of Update 4.", ""]
if first:
    v = first["versions"]
    lines += [
        "## Setup", "",
        f"- **Questions:** BFCL `non_live`, {first['n_questions']} questions per run, bfcl-eval {v['bfcl-eval']}",
        f"- **Prompt format:** `{first['prompt_format']}` (as in Updates 3-4), {MODE} decoding; Qwen3 with thinking off",
        f"- **Decoding:** greedy (temperature 0), seed {first['config']['sampling']['seed']}, "
        f"at most {first['config']['sampling']['max_tokens']} answer tokens",
    ]
    if int8:
        q = int8["quantized_checkpoint"]
        lines.append(
            f"- **INT8:** LLM.int8() from bitsandbytes {q['versions']['bitsandbytes']}: 8-bit weights with one scale "
            f"per row; input features above {q['settings']['llm_int8_threshold']} stay in FP16. The checkpoint is made "
            f"with transformers {q['versions']['transformers']} (`src/quantize.py`); vLLM runs it in eager mode.")
    lines += [
        f"- **NF4:** vLLM quantizes the FP16 weights while loading: 4-bit NormalFloat, blocks of 64, double "
        f"quantization (bitsandbytes {v['bitsandbytes']})",
        f"- **Inference:** vLLM {v['vllm']} on one {first['gpus'][0]} per run (FP16 Qwen3-8B used two)",
        f"- **Software:** Python {v['python']}, torch {v['torch']}, transformers {v['transformers']}",
        f"- **Code:** commit `{(first['code']['commit'] or 'unknown')[:7]}`; model revisions are in `results/meta/`",
        "- **Overall** = BFCL's Non-Live Overall Acc (unweighted mean of simple, multiple, parallel and "
        "parallel-multiple; simple = mean of Python, Java, JavaScript). Irrelevance is reported on its own.",
        "",
        "## Overall accuracy (%)", "",
    ] + md_table(compare) + [
        "", "## Accuracy per category (%)", "",
    ] + md_table(accuracy) + ["", "## Format, length and speed", ""] + md_table(info)
failed = [r for r, s in status.items() if s != "OK"]
lines += ["", "## Notes", ""]
lines += [f"- Did not finish: {', '.join(failed)}" if failed else "- All runs finished."]
lines += ["- "]
report = "\n".join(lines) + "\n"
with open(f"{WORK}/{REPORT_NAME}", "w") as f:
    f.write(report)
print(report)

pack = f"{WORK}/update5_results"
shutil.rmtree(pack, ignore_errors=True)
shutil.copytree(f"{REPO}/results", f"{pack}/results", ignore=shutil.ignore_patterns("bfcl_repro", "figures"))
if os.path.exists(CHECK_DIR):
    shutil.copytree(CHECK_DIR, f"{pack}/check")
for name in [REPORT_NAME, "int8_nf4_overall.csv", "int8_nf4_accuracy.csv", "int8_nf4_run_info.csv",
             "requirements-freeze.txt"]:
    if os.path.exists(f"{WORK}/{name}"):
        shutil.copy(f"{WORK}/{name}", pack)
for log_path in sorted(glob.glob(f"{WORK}/*.log")):
    shutil.copy(log_path, pack)
archive = shutil.make_archive(pack, "zip", pack)
print("Saved", archive)

# INT8 and NF4 (Update 5)

**Date:** 2026-10-29
**Goal:** function-calling accuracy of every core model in INT8 and NF4 (bitsandbytes), compared with the FP16 baselines of Update 4.

## Setup

- **Questions:** BFCL `non_live`, 1390 questions per run, bfcl-eval 2026.3.23
- **Prompt format:** `json-array-v1` (as in Updates 3-4), free decoding; Qwen3 with thinking off
- **Decoding:** greedy (temperature 0), seed 42, at most 4096 answer tokens
- **NF4:** vLLM quantizes the FP16 weights while loading: 4-bit NormalFloat, blocks of 64, double quantization (bitsandbytes 0.45.5)
- **Inference:** vLLM 0.8.5 on one Tesla T4 per run (FP16 Qwen3-8B used two)
- **Software:** Python 3.11.17, torch 2.6.0, transformers 4.51.3
- **Code:** commit `6b6903c`; model revisions are in `results/meta/`
- **Overall** = BFCL's Non-Live Overall Acc (unweighted mean of simple, multiple, parallel and parallel-multiple; simple = mean of Python, Java, JavaScript). Irrelevance is reported on its own.

## Overall accura

## After the run: what goes into the repository

1. Download **`update5_results.zip`** from the notebook's **Output** section (or the saved version's Output tab).
2. Download the notebook with its outputs (**File → Download Notebook**) and replace `notebooks/04_int8_nf4.ipynb` with it.
3. From the zip:
   - `int8_nf4.md` → `results/int8_nf4.md`; write a few sentences under **Notes**
   - `results/meta/`, `results/scores/` and `results/summary.csv` → the same places in the repository
   - The raw answers in `results/raw/` stay out of git; keep the zip as a backup.
4. Add the GPU time to `notes/compute_log.md`, the INT8 and NF4 choices to `notes/decisions.md`, and the Update 5 entry at the top of `UPDATES.md`, then commit and push.